# Usage examples of the time-dependent dual TRILEX package

## N-point contour Green's functions

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
from itertools import product

from triqs.gf import MeshReTime  # TRIQS real time mesh
from triqs.gf import MeshProduct # A direct product of 1D meshes

from tddt.keldysh import KeldyshGF # Green's function container on a 2-branch Keldysh contour

### Construct some (zero) Keldysh Green's function objects

In [2]:
t_max = 5.0
N_t = 51

t_mesh = MeshReTime(0.0, t_max, N_t)  # A 1D real time grid

2-point contour functions

In [3]:
tt_mesh = MeshProduct(t_mesh, t_mesh) # A 2D mesh as a direct product of t_mesh with itself

# A scalar-valued function 
g_scalar = KeldyshGF(mesh=tt_mesh)

# A matrix-valued function
# Each of the two time arguments has a single discrete index (e.g. spin projection) associated with it.
# The index runs over two values 0, 1.
g_matrix = KeldyshGF(mesh=tt_mesh, arg_index_shapes=((2,), (2,)))

# A tensor-valued function
# Now, we have 2 indices associated with each time argument (e.g. spin projection and orbital index)
# Those two indices vary in the ranges 0, 1 and 0, 1, 2 correspondingly
g_tensor = KeldyshGF(mesh=tt_mesh, arg_index_shapes=((2, 3), (2, 3)))

2-point contour functions with an extra k-argument

In [4]:
# Import some TRIQS modules related to lattice
from triqs.lattice import BravaisLattice, BrillouinZone
from triqs.gf import MeshBrZone

lat = BravaisLattice(units=[(1, 0, 0), (0, 1, 0)])  # 2D square lattice
bz = BrillouinZone(lat)  # Brillouin zone of the lattice

n_k = 10 # Number of k-points along each dimension
bz_mesh = MeshBrZone(bz, n_k) # k-mesh on 1BZ

# A product of two real-time grids and a k-mesh
ttk_mesh = MeshProduct(t_mesh, t_mesh, bz_mesh)

g_k_scalar = KeldyshGF(mesh=ttk_mesh)
g_k_matrix = KeldyshGF(mesh=ttk_mesh, arg_index_shapes=((2,), (2,)))
g_k_tensor = KeldyshGF(mesh=ttk_mesh, arg_index_shapes=((2, 3), (2, 3)))

3-point contour functions (vertices)

In [5]:
ttt_mesh = MeshProduct(t_mesh, t_mesh, t_mesh)

# Scalar-valued 
v_scalar = KeldyshGF(mesh=ttt_mesh)

# One extra discrete index per time argument
v_1 = KeldyshGF(mesh=ttt_mesh, arg_index_shapes=((2,), (2,), (2,)))

# Two extra discrete indices per time argument
v_2 = KeldyshGF(mesh=ttt_mesh, arg_index_shapes=((2, 3), (2, 3), (2, 3)))

# The first two time arguments have 2 discrete indices attached to each of them.
# The last time argument has only one discrete index (e.g. the bosonic channel)
v_3 = KeldyshGF(mesh=ttt_mesh, arg_index_shapes=((2, 3), (2, 3), (4,)))

1-point contour functions, e.g. time-dependent dispersion

In [6]:
f_scalar = KeldyshGF(mesh=t_mesh)

# f_i(t), i=0,1
f_i = KeldyshGF(mesh=t_mesh, arg_index_shapes=((2,),))

# f_{ij}(t), i=0,1, j =0, ..., 4
f_ij = KeldyshGF(mesh=t_mesh, arg_index_shapes=((2, 5),))

Building a matrix/tensor-valued contour function out of scalar-valued components using `KeldyshGF.from_arg_index_gen()`.

In [7]:
tt_mesh = MeshProduct(t_mesh, t_mesh)

arg_index_shapes=((2, 3), # 2 spin projections, 3 orbitals 
                  (2, 3)) # 2 spin projections, 3 orbitals

# Generator of scalar-valued elements
def generator(ind1, ind2):
    sigma1, orb1 = ind1
    sigma2, orb2 = ind2

    # Return a zero scalar-valued KeldyshGF object in the spin-offdiagonal case
    if sigma1 != sigma2:
        return KeldyshGF(mesh=tt_mesh)
    else:
        # Otherwise build the corresponding scalar-valued element e.g. by calling an impurity solver
        g_el = KeldyshGF(mesh=tt_mesh)
        # Fill g_el ...
        return g_el

# Build a GF that is diagonal in spin indices
G = KeldyshGF.from_arg_index_gen(generator, mesh=tt_mesh, arg_index_shapes=arg_index_shapes)

### Access real-time components of `KeldyshGF`

In [8]:
from tddt.keldysh import Branch

FW = Branch.FORWARD
BW = Branch.BACKWARD

# 2-point GF
print(g_matrix[FW, FW])  # Keldysh component g^{++}(t, t') as a TRIQS Gf object defined on MeshProduct(t_mesh, t_mesh)
print(g_matrix[FW, BW])  # Keldysh component g^{+-}(t, t')

# Vertex
print(v_3[FW, BW, BW])   # Keldysh component v^{+--}(t, t', t'')

Greens Function  with mesh Real Time Mesh with t_min = 0, t_max = 5, n_t = 51, Real Time Mesh with t_min = 0, t_max = 5, n_t = 51 and target_shape (2, 2): 

Greens Function  with mesh Real Time Mesh with t_min = 0, t_max = 5, n_t = 51, Real Time Mesh with t_min = 0, t_max = 5, n_t = 51 and target_shape (2, 2): 

Greens Function  with mesh Real Time Mesh with t_min = 0, t_max = 5, n_t = 51, Real Time Mesh with t_min = 0, t_max = 5, n_t = 51, Real Time Mesh with t_min = 0, t_max = 5, n_t = 51 and target_shape (2, 3, 2, 3, 4): 



Extract $G^<(t,t')$, $G^>(t,t')$, $G^{ret}(t,t')$ and $G^{adv}(t,t')$

In [9]:
print(g_tensor.lesser())
print(g_tensor.greater())
print(g_tensor.retarded())
print(g_tensor.advanced())

Greens Function  with mesh Real Time Mesh with t_min = 0, t_max = 5, n_t = 51, Real Time Mesh with t_min = 0, t_max = 5, n_t = 51 and target_shape (2, 3, 2, 3): 

Greens Function  with mesh Real Time Mesh with t_min = 0, t_max = 5, n_t = 51, Real Time Mesh with t_min = 0, t_max = 5, n_t = 51 and target_shape (2, 3, 2, 3): 

Greens Function  with mesh Real Time Mesh with t_min = 0, t_max = 5, n_t = 51, Real Time Mesh with t_min = 0, t_max = 5, n_t = 51 and target_shape (2, 3, 2, 3): 

Greens Function  with mesh Real Time Mesh with t_min = 0, t_max = 5, n_t = 51, Real Time Mesh with t_min = 0, t_max = 5, n_t = 51 and target_shape (2, 3, 2, 3): 



Access a value corresponding to a single mesh point.

In [10]:
from tddt.keldysh import ContourPoint

# Points on the real time axis 
t_points = list(t_mesh)
t1, t2 = t_points[0], t_points[1]

# Contour points
z1 = ContourPoint(Branch.FORWARD, t1)
z2 = ContourPoint(Branch.BACKWARD, t2)

print(g_scalar[z1, z2])
print(g_matrix[z1, z2])

# Set a single value in a matrix-valued GF
g_matrix[z1, z2] = np.eye(2)

print(g_matrix[z1, z2][0, 0])
print(g_matrix[z1, z2][0, 1])

0j
[[0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j]]
(1+0j)
0j


### Factory functions for `KeldyshGF`

Construct a 2-point contour function from its lesser and greater components.

In [11]:
from triqs.gf import Gf
from tddt.keldysh import KeldyshGF

from scipy.linalg import expm  # Matrix exponential

H = np.array([[1.0, 0.5j], [-0.5j, 2.0]])
n = 0.1

g_l = Gf(mesh=tt_mesh, target_shape=(2, 2))
g_g = Gf(mesh=tt_mesh, target_shape=(2, 2))

for t1, t2 in tt_mesh:
    g_g[t1, t2] = -1j * (1.0 - n) * expm(-1j * H * (t1 - t2))
    g_l[t1, t2] = -1j * (-n) * expm(-1j * H * (t1 - t2))

g = KeldyshGF.from_lesser_greater(g_l, g_g)

Construct a 3-point vertex from 6 real time correlators.

Each element of dictionary G corresponds to one permutation of operators
in the correlator,
$$
    G_{ijk}(t_0, t_1, t_2) = -\xi_{ijk} \langle O_i(t_i) O_j(t_j) O_k(t_k)\rangle,
$$

where $O_0(t_0) = c(t_0)$, $O_1(t_1) = c^\dagger(t_1)$, $O_2(t_2) = \rho(t_2)$.
$\xi_{ijk} = -1$ if permutation (ijk) swaps indices 0 and 1, and +1 otherwise.

Keys are 3! = 6 triplets (i, j, k), which are permutations of (0, 1, 2) indicating the respective order of $c$, $c^\dagger$ and $\rho$.

In [12]:
from tddt.keldysh import KeldyshGF

def make_time_piece(x):
    g = Gf(mesh=ttt_mesh, target_shape=())
    g.data[:] = x
    return g

G = {(0, 1, 2): make_time_piece(1.0),  # G_{012}(t_0, t_1, t_2)
     (0, 2, 1): make_time_piece(2.0),  # G_{021}(t_0, t_1, t_2)
     (1, 0, 2): make_time_piece(3.0),  # G_{102}(t_0, t_1, t_2)
     (1, 2, 0): make_time_piece(4.0),  # G_{120}(t_0, t_1, t_2)
     (2, 0, 1): make_time_piece(5.0),  # G_{201}(t_0, t_1, t_2)
     (2, 1, 0): make_time_piece(6.0)}  # G_{210}(t_0, t_1, t_2)

Lambda = KeldyshGF.from_vertex3_pieces(G)

### Transposition of a 2-point contour function.

Transposition of a 2-point contour function $G_{a,b}(z, z')$ equals, by definition, $G_{b,a}(z', z)$.

In [13]:
from triqs.gf import Gf
from tddt.keldysh import KeldyshGF

from scipy.linalg import expm  # Matrix exponential

H = np.array([[1.0, 0.5j], [-0.5j, 2.0]])
n = 0.1

g_l = Gf(mesh=tt_mesh, target_shape=(2, 2))
g_g = Gf(mesh=tt_mesh, target_shape=(2, 2))

for t1, t2 in tt_mesh:
    g_g[t1, t2] = -1j * (1.0 - n) * expm(-1j * H * (t1 - t2))
    g_l[t1, t2] = -1j * (-n) * expm(-1j * H * (t1 - t2))

g = KeldyshGF.from_lesser_greater(g_l, g_g)

# Transposition of 'g'
g_T = g.T 
g_T_l, g_T_g = g_T.lesser(), g_T.greater()

print(all((g_T_l[t1, t2] == g_g[t2, t1].T).all() for t1, t2 in tt_mesh))
print(all((g_T_g[t1, t2] == g_l[t2, t1].T).all() for t1, t2 in tt_mesh))

True
True


### Hermitian conjugate of a 2-point contour function.

$C^\ddagger_{a,b}(z, z')$ is a Hermitian conjugate of $C_{a,b}(z, z')$ if
$$
\begin{array}{ll}
    [C^\ddagger]^<_{a,b}(t, t') &= -[C^<_{b,a}(t', t)]^*, \\
    [C^\ddagger]^>_{a,b}(t, t') &= -[C^>_{b,a}(t', t)]^*, \\
    [C^\ddagger]^{ret}_{a,b}(t, t') &= [C^{adv}_{b,a}(t', t)]^*, \\
    [C^\ddagger]^{adv}_{a,b}(t, t') &= [C^{ret}_{b,a}(t', t)]^*
\end{array}
$$
$a$ and $b$ are multi-indices associated with $t$ and $t'$.

In [14]:
from tddt.keldysh import KeldyshGF, herm_conj

def make_keldysh_gf(H):
    g_l = Gf(mesh=tt_mesh, target_shape=(2, 2))
    g_g = Gf(mesh=tt_mesh, target_shape=(2, 2))
    
    n = 0.1
    for t1, t2 in tt_mesh:
        g_g[t1, t2] = -1j * (1.0 - n) * expm(-1j * H * (t1 - t2))
        g_l[t1, t2] = -1j * (-n) * expm(-1j * H * (t1 - t2))
        
    return KeldyshGF.from_lesser_greater(g_l, g_g)

#
# Prepare a non-Hermitian 2-point contour function
#

g = make_keldysh_gf(np.array([[1.0, 0.5j], [0, 2.0]]))
g_herm = make_keldysh_gf(np.array([[1.0, 0.5j], [-0.5j, 2.0]]))

print("g is Hermitian:", g.is_hermitian())
print("g_herm is Hermitian:", g_herm.is_hermitian())

g_l, g_g = g.lesser(), g.greater()
g_ret, g_adv = g.retarded(), g.advanced()

g_hc = herm_conj(g)

g_hc_l, g_hc_g = g_hc.lesser(), g_hc.greater()
g_hc_ret, g_hc_adv = g_hc.retarded(), g_hc.advanced()
        
print(all((g_hc_l[t1, t2] == -g_l[t2, t1].conj().T).all() for t1, t2 in tt_mesh))
print(all((g_hc_g[t1, t2] == -g_g[t2, t1].conj().T).all() for t1, t2 in tt_mesh))
print(all((g_hc_ret[t1, t2] == g_adv[t2, t1].conj().T).all() for t1, t2 in tt_mesh))
print(all((g_hc_adv[t1, t2] == g_ret[t2, t1].conj().T).all() for t1, t2 in tt_mesh))

g is Hermitian: False
g_herm is Hermitian: True
True
True
True
True


Extended retarded and advanced parts have step functions omitted from their definitions.

$$
\begin{array}{ll}
    G^{ret,e}(t, t') &= G^>(t, t') - G^<(t, t'),\\
    G^{adv,e}(t, t') &= G^<(t, t') - G^>(t, t').
\end{array}
$$

With help of these auxiliary functions, the Hermitian conjugation is defined in a more straightforward way.
$$
\begin{array}{ll}
    [C^\ddagger]^<_{a,b}(t, t') &= -[C^<_{b,a}(t', t)]^*, \\
    [C^\ddagger]^>_{a,b}(t, t') &= -[C^>_{b,a}(t', t)]^*, \\
    [C^\ddagger]^{ret,e}_{a,b}(t, t') &= -[C^{ret,e}_{b,a}(t', t)]^*, \\
    [C^\ddagger]^{adv,e}_{a,b}(t, t') &= -[C^{adv,e}_{b,a}(t', t)]^*
\end{array}
$$

In [15]:
g_ret_ext, g_adv_ext = g.retarded_ext(), g.advanced_ext()
g_hc_ret_ext, g_hc_adv_ext = g_hc.retarded_ext(), g_hc.advanced_ext()

print(all((g_hc_ret_ext[t1, t2] == -g_ret_ext[t2, t1].conj().T).all() for t1, t2 in tt_mesh))
print(all((g_hc_adv_ext[t1, t2] == -g_adv_ext[t2, t1].conj().T).all() for t1, t2 in tt_mesh))

True
True


### Arithmetics of `KeldyshGF`

In [16]:
print(g + g)
print(g - g)
print(-g)
print(2 * g)

print(g == g)
print(g == -g)

True
False


### Contraction of discrete indices with an arbitrary tensor

Make a Green's function $G_{ij,kl}(t, t')$ and an array $U_{ijkl}$.

In [17]:
G = KeldyshGF(mesh=tt_mesh, arg_index_shapes=((2, 3), (2, 3)))    
U = np.ones((2, 3, 3, 2))
print(G.arg_index_shapes[0])

(2, 3)


Compute a discrete index contraction $F_{ij,kl}(t, t') = \sum_{mn} U_{ij,nm} G_{mn,kl}(t, t')$.

In [18]:
from tddt.keldysh import target_dot

F = target_dot(G,
               U,
               0,     # Contract over all indices associated with the first time argument of G, i.e. m and n
               (3, 2) # Contract over the 4th and the 3rd indices of U, in this specific order
               )
print(F)

## Convolutions on the contour

Contour convolution of two 2-point functions together with discrete index contraction,
$$
    C_{ij,kl}(t, t') = \sum_{mn} \int_\mathcal{C} d\bar t A_{ij,mn}(t, \bar t) B_{mn,kl}(\bar t, t').
$$
The high precision algorithm used by operator `@` for contour integration is described in Section 11 of

```
NESSi: The Non-Equilibrium Systems Simulation package,
M. Schüler, D. Golež, Y. Murakami, N. Bittner, A. Herrmann, H. U.R. Strand, P. Werner, M. Eckstein,
Computer Physics Communications,
Volume 257, 2020, 107484,
https://doi.org/10.1016/j.cpc.2020.107484.
```

In [19]:
# i = 0, 1
# j = 0, 1, 2
# m = 0, 1
# n = 0, 1, 2, 3
# k = 0, 1
# l = 0, 1, 2, 3, 4
A = KeldyshGF(mesh=tt_mesh, arg_index_shapes=((2, 3), (2, 4)))
B = KeldyshGF(mesh=tt_mesh, arg_index_shapes=((2, 4), (2, 5)))

C = A @ B

print(C.n_args)  # Number of time arguments
print(C.arg_index_shapes)

2
((2, 3), (2, 5))


Simultaneous convolution of two 3-point objects over two time arguments and contraction of the respective discrete indices,
$$
    C_{ij,kl}(t, t') = \sum_{mn}\sum_{pq} \int_\mathcal{C} dt_1 dt_2 A_{mn,ij,pq}(t_1, t, t_2) B_{pq,kl,mn}(t_2, t', t_1).
$$
Current implementation of `conv()` gives inaccurate results with absolute error scaling linearly with the time mesh step.

A proper implementation of it could be based on the multi-point contour calculus described in 
```
Contour calculus for many-particle functions,
M. J. Hyrkäs, D. Karlsson, R. van Leeuwen,
Journal of Physics A: Mathematical and Theoretical
Volume 52, Number 21, 2019, 215303
https://iopscience.iop.org/article/10.1088/1751-8121/ab165d.
```

In [20]:
from tddt.keldysh import conv

mesh = MeshProduct(MeshReTime(0.0, t_max, 11),
                   MeshReTime(0.0, t_max, 11),
                   MeshReTime(0.0, t_max, 11))

A = KeldyshGF(mesh=mesh, arg_index_shapes=((2, 3), (2, 4), (2, 5)))
B = KeldyshGF(mesh=mesh, arg_index_shapes=((2, 5), (2, 6), (2, 3)))

C = conv(A, B,
         [(0, 2), (2, 0)])  # Coupling of arguments of A and B: 0 <--> 2, 2 <--> 0 

print(C.n_args)  # Number of time arguments
print(C.arg_index_shapes)

2
((2, 4), (2, 6))


Handling of the non-time mesh components (e.g. momentum/lattice site argument) by `conv()` and `@`:

- If non-time mesh components of `A` and `B` argee, then `C` has the same non-time mesh components.
- Otherwise `C` is defined on a direct product of `A`'s and `B`'s non-time mesh components.

In [21]:
bz_mesh1 = MeshBrZone(bz, 3)
bz_mesh2 = MeshBrZone(bz, 4)

# A and B are defined on the same k-mesh
A = KeldyshGF(mesh=MeshProduct(t_mesh, t_mesh, bz_mesh1), arg_index_shapes=((2, 2), (2, 2)))
B = KeldyshGF(mesh=MeshProduct(t_mesh, t_mesh, bz_mesh1), arg_index_shapes=((2, 2), (2, 2)))

C = A @ B
print(len(C.non_time_mesh.components))

# A and B are defined on different k-meshes
A = KeldyshGF(mesh=MeshProduct(t_mesh, t_mesh, bz_mesh1), arg_index_shapes=((2, 2), (2, 2)))
B = KeldyshGF(mesh=MeshProduct(t_mesh, t_mesh, bz_mesh2), arg_index_shapes=((2, 2), (2, 2)))

C = A @ B
print(len(C.non_time_mesh.components))

1
2


## Singular 2-point Green's functions

There is a special class `Singular2PKeldyshGF` representing a 2-point contour function of the form
$$
    G_{a,b}(t, t') = g_{a,b}(t) \delta_\mathcal{C}(t, t'),
$$
where $a$ and $b$ are multi-indices associated with $t$ and $t'$ respectively, and $\delta_\mathcal{C}$ is a Dirac delta function on the contour. `Singular2PKeldyshGF` conserves memory by storing only two components $g^+_{a,b}(t)$ and $g^-_{a,b}(t)$ defined on a one-dimensional real-time mesh instead of the 4 components $G^{++}_{a,b}(t, t')$, $G^{+-}_{a,b}(t, t')$, $G^{-+}_{a,b}(t, t')$ and $G^{--}_{a,b}(t, t')$, each defined on a 2D real-time mesh.

In [22]:
from tddt.keldysh import Singular2PKeldyshGF

# A scalar-valued function 
g_s2p_scalar = Singular2PKeldyshGF(mesh=t_mesh)

# A matrix-valued function
# Each of the two time arguments has a single discrete index (e.g. spin projection) associated with it.
# The index runs over two values 0, 1.
g_s2p_matrix = Singular2PKeldyshGF(mesh=t_mesh, arg_index_shapes=((2,), (2,)))

# A tensor-valued function
# Now, we have 2 indices associated with each time argument (e.g. spin projection and orbital index)
# Those two indices vary in the ranges 0, 1 and 0, 1, 2 correspondingly
g_s2p_tensor = Singular2PKeldyshGF(mesh=t_mesh, arg_index_shapes=((2, 3), (2, 3)))

The factory class method `Singular2PKeldyshGF.from_retime()` offers a convenient way to construct a singular GF from a real-time function $g_{a,b}(t) = g^+_{a,b}(t) = g^-_{a,b}(t)$.

In [23]:
# Import some TRIQS modules related to lattice
from triqs.lattice import BravaisLattice, BrillouinZone
from triqs.gf import Gf, MeshBrZone

lat = BravaisLattice(units=[(1, 0, 0), (0, 1, 0)])  # 2D square lattice
bz = BrillouinZone(lat)  # Brillouin zone of the lattice

n_k = 10 # Number of k-points along each dimension
bz_mesh = MeshBrZone(bz, n_k) # k-mesh on 1BZ

# A product of a real-time grid and a k-mesh
tk_mesh = MeshProduct(t_mesh, bz_mesh)

# Make a time- and spin-dependent dispersion function $\epsilon_k(t)$
eps_tk = Gf(mesh=tk_mesh, target_shape=(2, 2))
for t, k in tk_mesh:
    eps_tk[t, k] = -2 * np.array([[[1, 0], [0, -1]]]) * np.cos(t.value) * (np.cos(k[0]) + np.cos(k[1]))

# Turn eps_tk into a singular 2-point function $\epsilon_k(t)\delta_C(t, t')$
eps_s2p = Singular2PKeldyshGF.from_retime(eps_tk)

print("Mesh of eps_s2p:\n", eps_s2p.mesh)
print("Real time mesh component of eps_s2p:\n", eps_s2p.time_mesh)
print("arg_index_shapes of eps_s2p:\n", eps_s2p.arg_index_shapes)

Mesh of eps_s2p:
 Real Time Mesh with t_min = 0, t_max = 5, n_t = 51, Brillouin Zone Mesh with linear dimensions (10 10 1)
 -- units = 
[[0.628319,0,0]
 [0,0.628319,0]
 [0,0,6.28319]]
 -- brillouin_zone: Brillouin Zone with 2 dimensions and reciprocal matrix 
[[6.28319,0,0]
 [0,6.28319,0]
 [0,0,6.28319]]
Real time mesh component of eps_s2p:
 Real Time Mesh with t_min = 0, t_max = 5, n_t = 51
arg_index_shapes of eps_s2p:
 ((2,), (2,))


### Convolutions involving singular 2-point functions

`Singular2PKeldyshGF` objects can participate in contour convolutions.

If
$A_{i,j}(t, t') = a_{i,j}(t)\delta_\mathcal{C}(t,t')$, $B_{i,j}(t, t') = b_{i,j}(t)\delta_\mathcal{C}(t,t')$ and $C_{i,j}(t, t') = c_{i,j}(t)\delta_\mathcal{C}(t,t')$, then
$$
    C_{i,j}(t, t') = \sum_{k} \int_\mathcal{C} d\bar t A_{i,k}(t, \bar t) B_{k,j}(\bar t, t') \quad\Rightarrow\quad
    c_{i,j}(t) = \sum_{k} a_{i,k}(t) b_{k,j}(t).
$$

In [24]:
from tddt.keldysh import Branch

a_t = Gf(mesh=t_mesh, target_shape=(2, 2))
b_t = Gf(mesh=t_mesh, target_shape=(2, 2))
for t in t_mesh:
    a_t[t] = np.array([[[1, 2], [3, 4]]]) * np.cos(t.value)
    b_t[t] = np.array([[[5, 6], [7, 8]]]) * np.sin(t.value)

A_s2p = Singular2PKeldyshGF.from_retime(a_t)
B_s2p = Singular2PKeldyshGF.from_retime(b_t)

C_s2p = A_s2p @ B_s2p

# Check result of the convolution
for t in C_s2p.time_mesh:
    c_fw = C_s2p[Branch.FORWARD]
    c_bw = C_s2p[Branch.BACKWARD]
    assert np.allclose(c_fw[t], np.array([[19, 22], [43, 50]]) * np.cos(t.value) * np.sin(t.value))
    assert np.allclose(c_bw[t], np.array([[19, 22], [43, 50]]) * np.cos(t.value) * np.sin(t.value))

One can mix arguments of types `KeldyshGF` and `Singular2PKeldyshGF` in calls to the operator `@` and function `conv()`.
The latter supports convolutions over one or two pairs of arguments in this case.

In [25]:
# Prepare arguments
from triqs.gf import Gf
from tddt.keldysh import KeldyshGF, Singular2PKeldyshGF

from scipy.linalg import expm  # Matrix exponential

# Construct a regular 2-point GF
H = np.array([[1.0, 0.5j], [-0.5j, 2.0]])
n = 0.1

g_l = Gf(mesh=tt_mesh, target_shape=(2, 2))
g_g = Gf(mesh=tt_mesh, target_shape=(2, 2))

for t1, t2 in tt_mesh:
    g_g[t1, t2] = -1j * (1.0 - n) * expm(-1j * H * (t1 - t2))
    g_l[t1, t2] = -1j * (-n) * expm(-1j * H * (t1 - t2))

G = KeldyshGF.from_lesser_greater(g_l, g_g)

# Construct a singular 2-point GF
S_t = Gf(mesh=t_mesh, target_shape=(2, 2))

S_mat = np.array([[[1, 0.25], [0.25, 1]]])
for t in t_mesh:
    S_t[t] = S_mat * np.cos(t.value)

S_s2p = Singular2PKeldyshGF.from_retime(S_t)

In [26]:
# Compute convolutions
GS = G @ S_s2p
SG = S_s2p @ G

# Check results
GS_l, GS_g = GS.lesser(), GS.greater()
SG_l, SG_g = SG.lesser(), SG.greater()

for t1, t2 in tt_mesh:
    # G @ S_s2p
    assert np.allclose(GS_l[t1, t2], -1j * (-n) * expm(-1j * H * (t1 - t2)) @ (S_mat * np.cos(t2.value)))
    assert np.allclose(GS_g[t1, t2], -1j * (1.0 - n) * expm(-1j * H * (t1 - t2)) @ (S_mat * np.cos(t2.value)))
    # S_s2p @ G
    assert np.allclose(SG_l[t1, t2], (S_mat * np.cos(t1.value)) @ (-1j * (-n) * expm(-1j * H * (t1 - t2))))
    assert np.allclose(SG_g[t1, t2], (S_mat * np.cos(t1.value)) @ (-1j * (1.0 - n) * expm(-1j * H * (t1 - t2))))

In [27]:
from tddt.keldysh import conv

# Use conv() to compute convolutions with a non-standard order of time arguments
GS = conv(G, S_s2p, [(0, 0)]) # \int dt G(t, t_1) S(t, t_2)
SG = conv(S_s2p, G, [(1, 1)]) # \int dt S(t_1, t) G(t_2, t)

# Check results
GS_l, GS_g = GS.lesser(), GS.greater()
SG_l, SG_g = SG.lesser(), SG.greater()

for t1, t2 in tt_mesh:
    # conv(G, S_s2p, [(0, 0)])
    assert np.allclose(GS_l[t1, t2], -1j * (1.0 - n) * expm(-1j * H * (t2 - t1)).T @ (S_mat * np.cos(t2.value)))
    assert np.allclose(GS_g[t1, t2], -1j * (-n) * expm(-1j * H * (t2 - t1)).T @ (S_mat * np.cos(t2.value)))
    # conv(S_s2p, G, [(1, 1)])
    assert np.allclose(SG_l[t1, t2], (S_mat * np.cos(t1.value)) @ (-1j * (1.0 - n) * expm(-1j * H * (t2 - t1)).T))
    assert np.allclose(SG_g[t1, t2], (S_mat * np.cos(t1.value)) @ (-1j * (-n) * expm(-1j * H * (t2 - t1)).T))

## 2-nd order dual diagrams for the self-energy and the polarization operator

3-point vertex $\Lambda_{\sigma_1 l_1,\sigma_2 l_2}^{\varsigma l_3 l_4}(t, t', t'')$, fermionic line $G_{\sigma_1 l_1, \sigma_2 l_2}(t, t'; \mathbf{r})$ and bosonic line $W_{\varsigma l_1 l_2, \varsigma' l_3 l_4}(t, t'; \mathbf{r})$.

Here, we use a periodic real-space mesh to turn $\mathbf{k}$- and $\mathbf{q}$-summations in the diagrams into products.

In [28]:
from triqs.gf import MeshCycLat

t_mesh = MeshReTime(0, 5.0, 11)

# Vertex
arg_index_shapes = ((2, 3),    # \sigma_1, l_1
                    (2, 3),    # \sigma_2, l_2
                    (4, 3, 3)) # \varsigma, l_3, l_4

Lambda = KeldyshGF(mesh=MeshProduct(t_mesh, t_mesh, t_mesh), arg_index_shapes=arg_index_shapes)

n_r = 3  # Number of r-mesh points in each spacial direction
r_mesh = MeshCycLat(lat, n_r)

# Fermionic line
arg_index_shapes = ((2, 3),  # \sigma_1, l_1
                    (2, 3))  # \sigma_2, l_2
G = KeldyshGF(mesh=MeshProduct(t_mesh, t_mesh, r_mesh),
              arg_index_shapes=arg_index_shapes)

# Bosonic line
arg_index_shapes = ((4, 3, 3),  # \varsigma, l_1, l_2
                    (4, 3, 3))  # \varsigma', l_3, l_4
W = KeldyshGF(mesh=MeshProduct(t_mesh, t_mesh, r_mesh),
              arg_index_shapes=arg_index_shapes)

# q=0 component of the bosonic line
W_q0 = KeldyshGF(mesh=MeshProduct(t_mesh, t_mesh),
                 arg_index_shapes=arg_index_shapes)

In [29]:
from tddt.dtrilex import *

Pi = polarization_2nd_order(Lambda, G)              # Eq. (48) in Zhenya's notes
Sigma = selfenergy_2nd_order(Lambda, G, W)          # First line of Eq. (47) in Zhenya's notes
Sigma_HF = selfenergy_2nd_order_hf(Lambda, G, W_q0) # Second line of Eq. (47) in Zhenya's notes

<span style="color:red">N.B. As these functions rely on `tddt.keldysh.conv()`, they currently yield inaccurate results.</span>

## Contour Dyson-like equation in integral form

`tddt.vie2.solve_vie2()` solves the contour Dyson-like equation in integral form,
$$
    G(t, t') + \int_\mathcal{C} d\bar t F(t, \bar t) G(\bar t, t') = Q(t, t')
$$
under the additional assumptions $Q(t, t') = Q^\ddagger(t, t')$ and

$$
    \int_\mathcal{C} d\bar t F(t, \bar t) Q(\bar t, t') = \int_\mathcal{C} d\bar t Q(t, \bar t) F^\ddagger(\bar t, t').
$$
The contour convolutions imply summations over respective sets of discrete indices. Non-time mesh components of $Q$ and $F$ must agree.

The resulting $G(t, t')$ is also Hermitian, $G(t, t') = G^\ddagger(t, t')$.

In [30]:
t_mesh = MeshReTime(0.0, 5.0, 101)

n_k = 3
bz_mesh = MeshBrZone(bz, n_k)

mesh = MeshProduct(t_mesh, t_mesh, bz_mesh)

# Pauli matrices
s0 = np.eye(2, dtype=complex)
sx = np.array([[0, 1], [1, 0]], dtype=complex)
sy = np.array([[0, -1j], [1j, 0]], dtype=complex)

# Some simple linear dispersion law \eps(k)
eps_k = np.array([k.value[0] / np.pi + 1.0 for k in bz_mesh])
delta = 1.0

# Fill F(t, t') and Q(t, t')

F_g = Gf(mesh=mesh, target_shape=(2, 2))
F_l = Gf(mesh=mesh, target_shape=(2, 2))
Q_g = Gf(mesh=mesh, target_shape=(2, 2))
Q_l = Gf(mesh=mesh, target_shape=(2, 2))

from scipy.linalg import expm

for k, eps in zip(bz_mesh, eps_k):
    for t1, t2 in MeshProduct(t_mesh, t_mesh):
        dt = t1 - t2
        
        # Define Q via Hamiltonian H(k) = \eps(k) \sigma_0
        H = eps * s0
        e_Q = expm(-1j * H * dt)
        Q_g[t1, t2, k] = -1j * (1 - 0.1) * e_Q
        Q_l[t1, t2, k] = -1j * (-0.1) * e_Q
        
        # Define F via Hamiltonian H = \Delta * (\sigma_x + \sigma_y) / sqrt(2)
        H = delta * (sx + sy) / np.sqrt(2)
        e_F = expm(-1j * H * dt)
        F_g[t1, t2, k] = -1j * (1 - 0.2) * e_F
        F_l[t1, t2, k] = -1j * (-0.2) * e_F

F = KeldyshGF.from_lesser_greater(F_l, F_g)
Q = KeldyshGF.from_lesser_greater(Q_l, Q_g)

# Solve the equation

from tddt.vie2 import solve_vie2

G = solve_vie2(F, Q)

## Objects describing specific physical systems

Module `tddt.models` contains a few classes that represent finite physical systems and can be used to simplify construction of the respective Hamiltonians and Green's functions.

### `SingleFermion` and `FermionBand`

In [31]:
from tddt.models import SingleFermion

# Single fermionic degree of freedom with a given energy \epsilon
eps = 1.0
sf = SingleFermion(eps)

# Single-particle Green's function computed for a fixed occupation 'n'
n = 0.4
g = sf.gf(t_mesh, n=n)
print(g)

# Single-particle Green's function computed at a fixed temperature 'T'.
# The occupation of the state is derived from the Fermi-Dirac distribution in this case.
T = 3.0
g = sf.gf(t_mesh, T=T)
print(g)

In [32]:
from triqs.lattice import BravaisLattice, BrillouinZone
from triqs.gf import MeshBrZone

from tddt.models import FermionBand

lat = BravaisLattice(units=[(1, 0, 0), (0, 1, 0)])  # 2D square lattice
bz = BrillouinZone(lat)  # Brillouin zone of the lattice
bz_mesh = MeshBrZone(bz, 5)  # k-mesh on 1BZ

# Fermionic states forming a single band with dispersion law \epsilon(k)
def eps_k(k):
    return -2 * (np.cos(k[0]) + np.cos(k[1]))
fb = FermionBand(bz_mesh, eps_k)

# Single-particle Green's function computed for a k-dependent occupation n(k)
def n_k(k):
    return (k[0] + k[1]) / (4 * np.pi)
g = fb.gf(t_mesh, n_k=n_k)
print(g)

# Single-particle Green's function computed at a fixed temperature 'T'.
# The occupation of the states is derived from the Fermi-Dirac distribution in this case.
T = 3.0
g = fb.gf(t_mesh, T=T)
print(g)

### `FermionFlatBand`

In [33]:
from tddt.models import FermionFlatBand

# Fermionic band with a flat density of states of the half-bandwidth D centered at \omega = e0.
D = 2.0
e0 = 0.5
ffb = FermionFlatBand(D, e0)

# Single-particle Green's function computed at a fixed temperature 'T'.
T = 3.0
ffb.gf(t_mesh, T=T)

### `FiniteCluster`

In [34]:
# FiniteCluster describes electrons living on a finite collection of sites
from tddt.models import FiniteCluster

# Plaquette 2x2, Cartesian coordinates of the 4 sites
coords = [(0, 0, 0), (0, 1, 0), (1, 0, 0), (1, 1, 0)]

# Hopping matrix
# Its diagonal elements represent local energy levels.
mu = 1.0
t_nn = 0.3
t_nnn = -0.1
hopping = [[-mu,   t_nn,  t_nn,  t_nnn],
           [t_nn,  -mu,   t_nnn, t_nn],
           [t_nn,  t_nnn, -mu,   t_nn],
           [t_nnn, t_nn,  t_nn,  -mu]]

# Constants $U_i$ of local Coulomb interaction U_i n(\up, i) n(\dn, i)
U = 2.0
local_int=[U, U, U, U]

# Matrix $V_{ij}$ of non-local Coulomb interaction V_{ij} (n(\up, i) + n(\dn, i)) (n(\up, j) + n(\dn, j))
V = 0.2
nonlocal_int = [[0, V / 2, V / 2, 0],
                [V / 2, 0, 0, V / 2],
                [V / 2, 0, 0, V / 2],
                [0, V / 2, V / 2, 0]]

# Vector potential (x- and y-components). It is assumed spatially uniform.
A = (0.1, 0.2, 0)

# Construct a model object. All arguments apart from 'coords' are optional 
model = FiniteCluster(coords, hopping=hopping, local_int=local_int, nonlocal_int=nonlocal_int, vector_potential=A)

**Important:** Elements of the parameter arrays defined above can be either complex numbers or instances of `realevol.tinterp.TInterp` for the time-dependent quantities. It is also possible to mix both types of values within a single array.

In [35]:
# Fundamental operator set of 'model' -- can be passed to realevol.
print(model.fops)
# GF structure object describing a compatible TRIQS BlockGf container
print(model.gf_struct)
# Hamiltonian of the model
print(model.hamiltonian)

{('dn', 2), ('dn', 1), ('up', 1), ('up', 0), ('up', 3), ('dn', 0), ('dn', 3), ('up', 2)}
[('up', 4), ('dn', 4)]
(-0.0955336,-0.029552)*C^+(dn,0)C(dn,3) + (0.298501,0.02995)*C^+(dn,0)C(dn,2) + (0.29402,0.0596008)*C^+(dn,0)C(dn,1) + (-1,0)*C^+(dn,0)C(dn,0) + (0.298501,0.02995)*C^+(dn,1)C(dn,3) + (-0.0995004,0.00998334)*C^+(dn,1)C(dn,2) + (-1,0)*C^+(dn,1)C(dn,1) + (0.29402,-0.0596008)*C^+(dn,1)C(dn,0) + (0.29402,0.0596008)*C^+(dn,2)C(dn,3) + (-1,0)*C^+(dn,2)C(dn,2) + (-0.0995004,-0.00998334)*C^+(dn,2)C(dn,1) + (0.298501,-0.02995)*C^+(dn,2)C(dn,0) + (-1,0)*C^+(dn,3)C(dn,3) + (0.29402,-0.0596008)*C^+(dn,3)C(dn,2) + (0.298501,-0.02995)*C^+(dn,3)C(dn,1) + (-0.0955336,0.029552)*C^+(dn,3)C(dn,0) + (-0.0955336,-0.029552)*C^+(up,0)C(up,3) + (0.298501,0.02995)*C^+(up,0)C(up,2) + (0.29402,0.0596008)*C^+(up,0)C(up,1) + (-1,0)*C^+(up,0)C(up,0) + (0.298501,0.02995)*C^+(up,1)C(up,3) + (-0.0995004,0.00998334)*C^+(up,1)C(up,2) + (-1,0)*C^+(up,1)C(up,1) + (0.29402,-0.0596008)*C^+(up,1)C(up,0) + (0.29402,0

In [36]:
# Parameters of the model can be accessed and changed after construction

# Hopping
model.hopping[0, 0] = -1.5
print(model.hopping)
# Local interaction
model.local_int[1] = 3.0
print(model.local_int)
# Non-local interaction
model.nonlocal_int[0, 1] = model.nonlocal_int[1, 0] = 0.0
print(model.nonlocal_int)
# Vector potential
model.A = (0.2, 0.1, 0)
print(model.vector_potential)

[[-1.5  0.3  0.3 -0.1]
 [ 0.3 -1.  -0.1  0.3]
 [ 0.3 -0.1 -1.   0.3]
 [-0.1  0.3  0.3 -1. ]]
[2. 3. 2. 2.]
[[0.  0.  0.1 0. ]
 [0.  0.  0.  0.1]
 [0.1 0.  0.  0.1]
 [0.  0.1 0.1 0. ]]
(0.1, 0.2, 0)


It is possible to split all sites of a `FiniteCluster` object into two categories - impurity and bath sites - and request the hybridization function that describes effect of the bath sites on the impurity sites. A few additional requirements must be fulfilled.

- No local interactions on the bath sites.
- No non-local interactions involving the bath sites.
- No hoppings between two different bath sites.

In [37]:
# One impurity site and three bath sites
coords = [(0, 0, 0), (1, 0, 0), (2, 0, 0), (3, 0, 0)]

ed = 2.0               # Energy level of the impurity
eps = [-1.0, 0.0, 1.0] # Bath energy levels (must be time-independent to allow computation of hybridization function)
V = 0.5                # Hopping amplitude between the impurity and the bath sites

hopping = [[ed, V,      V,      V     ],
           [V,  eps[0], 0,      0     ],
           [V,  0,      eps[1], 0     ],
           [V,  0,      0,      eps[2]]]

# Interaction strength on the impurity
U = 2.0
local_int=[U, 0, 0, 0]

# Vector potential
A = (0.1, 0.2, 0)

# Model object
model = FiniteCluster(coords, hopping=hopping, local_int=local_int, vector_potential=A)

# Compute hybridization function 
Delta = model.hybridization(
    t_mesh,     # Real time mesh
    [0],        # List of impurity sites
    [1, 2, 3],  # List of bath sites
    T=0.1       # This temperature is used to compute occupations of bath sites
)

# ((spin index 1, impurity site index 1), (spin index 2, impurity site index 2))
print(Delta.arg_index_shapes)

((2, 1), (2, 1))


## Lattice-related utility functions

A few lattice-related utility functions are defined in `tddt.lattice`.

In the following example we construct a single-particle Green's function $G_{\sigma,\sigma'}(t_1, t_2, k)$.

In [38]:
# Import some TRIQS modules related to lattice
from triqs.lattice import BravaisLattice, BrillouinZone
from triqs.gf import Gf, MeshBrZone

lat = BravaisLattice(units=[(1, 0, 0), (0, 1, 0)])  # 2D square lattice
bz = BrillouinZone(lat)  # Brillouin zone of the lattice

n_k = 10 # Number of k-points along each dimension
bz_mesh = MeshBrZone(bz, n_k) # k-mesh on 1BZ

# A product of two real-time grids and a k-mesh
ttk_mesh = MeshProduct(t_mesh, t_mesh, bz_mesh)

# Make the greater and lesser components of the k-resolved Keldysh GF $G(t_1, t_2, k)$
mu = 2.0
beta = 5.0
print(f"Chemical potential {mu=}")
print(f"Inverse temperature {beta=}")

g_l = Gf(mesh=ttk_mesh, target_shape=(2, 2))
g_g = Gf(mesh=ttk_mesh, target_shape=(2, 2))
for t1, t2, k in ttk_mesh:
    # Dispersion \epsilon(k)
    eps_k = - mu - 2 * (np.cos(k[0]) + np.cos(k[1]))
    # Use Fermi distribution at T = 1 / beta for occupation
    occ = 1 / (1 + np.exp(beta * eps_k))
    # Make 'g' spin-diagonal
    S = np.array([[1.0, 0.0], [0.0, -1.0]])
    
    g_g[t1, t2, k] = -1j * S * (1.0 - occ) * np.exp(-1j * eps_k * (t1 - t2))
    g_l[t1, t2, k] = -1j * S * (-occ) * np.exp(-1j * eps_k * (t1 - t2))

g = KeldyshGF.from_lesser_greater(g_l, g_g)
print(g.mesh)
print(g.target_shape)

Chemical potential mu=2.0
Inverse temperature beta=5.0
Real Time Mesh with t_min = 0, t_max = 5, n_t = 101, Real Time Mesh with t_min = 0, t_max = 5, n_t = 101, Brillouin Zone Mesh with linear dimensions (10 10 1)
 -- units = 
[[0.628319,0,0]
 [0,0.628319,0]
 [0,0,6.28319]]
 -- brillouin_zone: Brillouin Zone with 2 dimensions and reciprocal matrix 
[[6.28319,0,0]
 [0,6.28319,0]
 [0,0,6.28319]]
(2, 2)


Function `tddt.lattice.local_part()` computes the average of its argument over each `MeshBrZone` mesh component the argument is defined on.

In [39]:
from tddt.keldysh import Branch

# Compute the local part of $G(t_1, t_2, k)$, i.e. an average over the Brillouin zone 
from tddt.lattice import local_part

g_loc = local_part(g)

# 'g_loc' is defined on the time meshes alone but has the same target shape as 'g'
print(g_loc.mesh)
print(g_loc.target_shape)

Real Time Mesh with t_min = 0, t_max = 5, n_t = 101, Real Time Mesh with t_min = 0, t_max = 5, n_t = 101
(2, 2)


`tddt.lattice.lattice_fourier()` performs a spacial Fourier transform of a `KeldyshGF` or `Singular2PKeldyshGF` object. Its keyword argument `apply_to` controls which spacial components of the object's mesh are targeted by the transform.
- `apply_to == tddt.lattice.SpacialArgs.BRZONE`: Apply transform to all `MeshBrZone` components of the mesh.
- `apply_to == tddt.lattice.SpacialArgs.LATTICE`: Apply transform to all `MeshCycLat` components of the mesh.
- `apply_to == tddt.lattice.SpacialArgs.BOTH`: Apply transform to both `MeshBrZone` and `MeshCycLat` components of the mesh.

Passing `flip_arg_sign=True` will also compute the Fourier image as a function of spacial arguments with a flipped sign ($\mathbf{r} \mapsto -\mathbf{r}, \mathbf{k} \mapsto -\mathbf{k}$).

In [40]:
from tddt.lattice import lattice_fourier, SpacialArgs

# Apply Fourier transform k -> r to 'g'
g_r = lattice_fourier(g, apply_to=SpacialArgs.BRZONE)
# Apply Fourier transform k -> -r to 'g'
g_mr = lattice_fourier(g, apply_to=SpacialArgs.BRZONE, flip_arg_sign=True)

print(g_r.mesh)
print(g_r.target_shape)

Real Time Mesh with t_min = 0, t_max = 5, n_t = 101, Real Time Mesh with t_min = 0, t_max = 5, n_t = 101, Cyclic Lattice Mesh with linear dimensions (10 10 1)
 -- units = 
[[1,0,0]
 [0,1,0]
 [0,0,1]]
 -- lattice: Bravais Lattice with dimension 2, units 
[[1,0,0]
 [0,1,0]
 [0,0,1]], n_orbitals 1
(2, 2)
